In [1]:
from pathlib import Path
import re

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display

# Notebookのフォルダ・プロジェクト直下・親フォルダのどこからでも実行できるようにする。
PROJECT_DIR = next(
    (candidate.resolve()
     for base in (Path.cwd(), *Path.cwd().parents)
     for candidate in (base, base / "streamer_python")
     if (candidate / "streamer_drift_diffusion").is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError("streamer_python project not found")
import sys
sys.path.insert(0, str(PROJECT_DIR))
from streamer_drift_diffusion.output import load_output_times

# --- 設定 ---
OUTPUT_DIR = Path("outputdata_chemistry2_ia88nm8i")
DT_SECONDS = 1.0e-12
FPS = 5
GIF_PATH = Path("streamer_progress.gif")

# 元のMATLABファイルと同じ色範囲
COLOR_LIMITS = {
    "ne": (14.0, 22.0),       # log10(ne [m^-3])
    "phi": (0.0, 4000.0),    # V
    "Ex": (0.0, 400.0),      # Td
    "Ey": (-500.0, 0.0),     # Td
}

# 相対パスはNotebookの場所ではなく、プロジェクト直下を基準にする。
OUTPUT_DIR = OUTPUT_DIR if OUTPUT_DIR.is_absolute() else PROJECT_DIR / OUTPUT_DIR
GIF_PATH = GIF_PATH if GIF_PATH.is_absolute() else PROJECT_DIR / GIF_PATH

print(f"CSV folder : {OUTPUT_DIR.resolve()}")
print(f"GIF output : {GIF_PATH.resolve()}")

CSV folder : C:\Users\daiko\Documents\Python_venvSystem\3.14system\PyAxiStreamer\pyaxistreamer-master\outputdata_chemistry2_ia88nm8i
GIF output : C:\Users\daiko\Documents\Python_venvSystem\3.14system\PyAxiStreamer\pyaxistreamer-master\streamer_progress.gif


In [2]:
def available_steps(prefix: str) -> set[int]:
    pattern = re.compile(rf"^{re.escape(prefix)}(\d+)\.csv$")
    result = set()
    for path in OUTPUT_DIR.glob(f"{prefix}*.csv"):
        match = pattern.match(path.name)
        if match:
            result.add(int(match.group(1)))
    return result

def load_c_matrix(prefix: str, step: int, nr: int, nz: int) -> np.ndarray:
    values = np.loadtxt(OUTPUT_DIR / f"{prefix}{step}.csv")
    expected = nr * nz
    if values.size != expected:
        raise ValueError(
            f"{prefix}{step}.csv has {values.size} values; expected {expected}. "
            "The Julia program may still be writing this file."
        )
    # Julia出力はC版互換の for(i) for(j) 順
    return values.reshape((nr, nz), order="C")

if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f"Output directory not found: {OUTPUT_DIR.resolve()}")

r_m = np.atleast_1d(np.loadtxt(OUTPUT_DIR / "rh.csv"))
z_m = np.atleast_1d(np.loadtxt(OUTPUT_DIR / "zh.csv"))
nr, nz = len(r_m), len(z_m)
prefixes = ("ne", "phi", "Ex", "Ey")

# 4種類すべてが揃っているstepだけを使用
steps = sorted(set.intersection(*(available_steps(prefix) for prefix in prefixes)))
if not steps:
    raise FileNotFoundError("No complete ne/phi/Ex/Ey frame set was found.")

frames = {
    prefix: [load_c_matrix(prefix, step, nr, nz) for step in steps]
    for prefix in prefixes
}

times_s = load_output_times(OUTPUT_DIR, steps, DT_SECONDS)

print(f"grid       : nr={nr}, nz={nz}")
print(f"frames     : {len(steps)}")
print(f"step range : {steps[0]} - {steps[-1]}")
print(f"time range : {times_s[0] * 1e9:.4g} - {times_s[-1] * 1e9:.4g} ns")

grid       : nr=400, nz=300
frames     : 16
step range : 0 - 15
time range : 0 - 1.5 ns


In [3]:
# 配列は (r, z) なので、画像表示用に (z, r) へ転置
plot_frames = {
    "ne": [np.log10(np.maximum(frame, 1.0e-1)).T for frame in frames["ne"]],
    "phi": [frame.T for frame in frames["phi"]],
    "Ex": [frame.T for frame in frames["Ex"]],
    "Ey": [frame.T for frame in frames["Ey"]],
}

extent = (
    float(r_m[0] * 1e3), float(r_m[-1] * 1e3),
    float(z_m[0] * 1e3), float(z_m[-1] * 1e3),
)
print(f"r range: {extent[0]:.4g} - {extent[1]:.4g} mm")
print(f"z range: {extent[2]:.4g} - {extent[3]:.4g} mm")

r range: 0.0025 - 1.998 mm
z range: 0.0025 - 1.497 mm


In [4]:
plt.rcParams["animation.embed_limit"] = 100.0

specs = [
    ("ne", r"Electron density $\log_{10}(n_e [\mathrm{m}^{-3}])$", r"$\log_{10}(n_e [\mathrm{m}^{-3}])$"),
    ("phi", r"Potential $\phi$", "V"),
    ("Ex", r"Radial field $E_r$", "Td"),
    ("Ey", r"Axial field $E_z$", "Td"),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 7.2), constrained_layout=True)
images = {}

for ax, (prefix, title, unit) in zip(axes.flat, specs):
    vmin, vmax = COLOR_LIMITS[prefix]
    image = ax.imshow(
        plot_frames[prefix][0],
        origin="lower", extent=extent, aspect="equal",
        cmap="jet", vmin=vmin, vmax=vmax, interpolation="nearest",
    )
    images[prefix] = image
    ax.set_title(title)
    ax.set_xlabel(r"Radial position $r$ [mm]")
    ax.set_ylabel(r"Axial position $z$ [mm]")
    fig.colorbar(image, ax=ax, shrink=0.82, label=unit)

time_text = fig.suptitle("")

def update(frame_index: int):
    step = steps[frame_index]
    time_ns = times_s[frame_index] * 1e9
    for prefix in prefixes:
        images[prefix].set_data(plot_frames[prefix][frame_index])
    time_text.set_text(f"step = {step}    time = {time_ns:.4f} ns")
    return (*images.values(), time_text)

animation = FuncAnimation(
    fig, update, frames=len(steps), interval=1000 / FPS, blit=False, repeat=True
)
plt.close(fig)
display(HTML(animation.to_jshtml(fps=FPS, default_mode="loop")))

In [5]:
animation.save(GIF_PATH, writer=PillowWriter(fps=FPS), dpi=120)
print(f"Saved: {GIF_PATH.resolve()} ({GIF_PATH.stat().st_size / 1e6:.2f} MB)")

# MP4保存にはffmpegが必要です。導入済みなら以下を有効化できます。
# from matplotlib.animation import FFMpegWriter
# animation.save("streamer_progress.mp4", writer=FFMpegWriter(fps=FPS), dpi=150)

Saved: C:\Users\daiko\Documents\Python_venvSystem\3.14system\PyAxiStreamer\pyaxistreamer-master\streamer_progress.gif (2.41 MB)
